In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
# os.environ["TOKENIZERS_PARALLELISM"] = "false"
# os.environ["MKL_SERVICE_FORCE_INTEL"]="1"
# os.environ["OMP_NUM_THREADS"] = "8"  # 控制MKL使用的线程数
# os.environ["MKL_NUM_THREADS"] = "8"
# os.environ["BLAS"] = "MKL"
# os.environ["USE_CUSOLVER"] = "1"

import numpy as np
import torch
from datasets import load_dataset
import evaluate
from transformers import (
    AutoFeatureExtractor, SwinForImageClassification,
    ViTImageProcessor, ViTForImageClassification,
    Trainer, TrainingArguments, EvalPrediction,
)
from peft import LoraConfig, get_peft_model
from trl import SFTConfig, SFTTrainer

from meft import MeftConfig, MeftTrainer
import meft

/home/shijx/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# import torch
# import torch._dynamo
# torch.set_float32_matmul_precision('high')
# torch._dynamo.config.disable = True
# torch._dynamo.config.suppress_errors = True
# torch._dynamo.config.cache_size_limit = 64
# torch._dynamo.config.recompile_limit = 64
# torch._dynamo.config.accumulated_cache_size_limit = 256

In [3]:
model_name = "google/vit-base-patch16-224-in21k"
processor = ViTImageProcessor.from_pretrained(model_name)
model = ViTForImageClassification.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    num_labels=100,
    ignore_mismatched_sizes=True,
    device_map="auto",
)

dataset = load_dataset("cifar100")

def transform(examples):
    inputs = processor(
        images=[x.convert("RGB") for x in examples["img"]],
        return_tensors="pt"
    )
    return {
        "pixel_values": inputs.pixel_values,
        "label": examples["fine_label"]
    }

dataset = dataset.with_transform(transform)

Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Using the latest cached version of the dataset since cifar100 couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'cifar100' at /home/shijx/.cache/huggingface/datasets/cifar100/cifar100/0.0.0/aadb3af77e9048adbea6b47c21a81e47dd092ae5 (last modified on Thu Jul  3 13:04:05 2025).


In [4]:
# model_name = "microsoft/swin-base-patch4-window7-224"
# feature_extractor = AutoFeatureExtractor.from_pretrained(model_name)
# model = SwinForImageClassification.from_pretrained(
#     model_name,
#     torch_dtype=torch.bfloat16,
#     num_labels=100,
#     ignore_mismatched_sizes=True,
#     device_map="auto",
# )

# dataset = load_dataset("cifar100")

# def transform(examples):
#     inputs = feature_extractor(
#         images=[x.convert("RGB") for x in examples["img"]],
#         return_tensors="pt"
#     )
#     return {
#         "pixel_values": inputs.pixel_values,
#         "label": examples["fine_label"]
#     }

# dataset = dataset.with_transform(transform)

In [5]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["query", "value"],
    bias="none",
    modules_to_save=["classifier"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 371,812 || all params: 86,247,368 || trainable%: 0.4311


In [6]:
def compute_metrics(eval_pred: EvalPrediction):
    evaluation = evaluate.load("accuracy")
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return evaluation.compute(predictions=predictions, references=labels)

In [7]:
trainer = MeftTrainer[Trainer](
    model=model,
    args=TrainingArguments(
        per_device_train_batch_size=32,
        gradient_accumulation_steps=4,
        per_device_eval_batch_size=32,
        num_train_epochs=1,
        learning_rate=1e-3,
        weight_decay=1e-2,
        lr_scheduler_type="cosine",
        bf16=True,
        bf16_full_eval=True,
        # deepspeed={
        #     "train_batch_size": "auto",
        #     "gradient_accumulation_steps": "auto",
        #     "zero_optimization": {
        #         "stage": 1,
        #     },
        # },
        use_liger_kernel=True,
        logging_steps=10,
        report_to="none",
        remove_unused_columns=False,
        label_names=["labels"],
    ),
    data_collator=None,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    compute_metrics=compute_metrics,
    meft_config=MeftConfig(
        patch_locations=(
            # "norm",
            # "attn_in",
            # "attn_out",
            # "mlp_in",
            # "mlp_out",
            # "act_fn",
            # "ckpt_attn",
            # "ckpt_mlp",
            "ckpt_layer",
        ),
        compress_kwargs={
            "rank": 128,
            "niter": 1,
        },
        # compress_workers=2,
    ),
)

Applying patch to vit model in: ('ckpt_layer',)


In [ ]:
trainer.train()

Step,Training Loss
10,4.478400
20,4.143300
30,3.712100
40,3.202500
50,2.670500
60,2.192700


In [ ]:
trainer.evaluate()

W0731 02:11:26.540000 881444 site-packages/torch/_dynamo/convert_frame.py:964] [0/8] torch._dynamo hit config.recompile_limit (8)
W0731 02:11:26.540000 881444 site-packages/torch/_dynamo/convert_frame.py:964] [0/8]    function: 'call_function' (/data2/shijx/code/Compress/meft/ops/checkpoint.py:47)
W0731 02:11:26.540000 881444 site-packages/torch/_dynamo/convert_frame.py:964] [0/8]    last reason: 0/7: GLOBAL_STATE changed: grad_mode 
W0731 02:11:26.540000 881444 site-packages/torch/_dynamo/convert_frame.py:964] [0/8] To log all recompilation reasons, use TORCH_LOGS="recompiles".
W0731 02:11:26.540000 881444 site-packages/torch/_dynamo/convert_frame.py:964] [0/8] To diagnose recompilation issues, see https://pytorch.org/docs/main/torch.compiler_troubleshooting.html.


{'eval_loss': 0.5671461224555969,
 'eval_accuracy': 0.8932,
 'eval_runtime': 29.9615,
 'eval_samples_per_second': 333.762,
 'eval_steps_per_second': 10.447,
 'epoch': 1.0}